# Analyse RPS Karasek

> **Notebook d'analyse des Risques Psychosociaux (RPS)** basé sur le modèle de Karasek.  
> Pipeline complet : chargement → nettoyage → scores → classification → visualisations → tableaux croisés.

---

### Note méthodologique

| Méthode       | Seuil                                | Usage                               |
| ------------- | ------------------------------------ | ----------------------------------- |
| **Théorique** | Point central de l'échelle (N × 2.5) | Dashboard — monitoring inter-vagues |

---

<br>

> **Règle** : Ne jamais écrire « à risque » — utiliser **« niveau supérieur au point central de l'échelle »**  
> Les scores **continus** sont toujours conservés en base (ne jamais sauvegarder uniquement des catégories).


## 1. Imports & Configuration


In [ ]:
# ── Dépendances standard ────────────────────────────────────────────────────
import logging
import unicodedata
import warnings
import textwrap
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── Calcul / données ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# ── Paramètres globaux ──────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── Répertoires de travail ──────────────────────────────────────────────────
for _d in ["../lib/data", "../reports", "../reports/karasek/logs", "../reports/karasek/figures", "../reports/karasek/tables"]:
    Path(_d).mkdir(exist_ok=True, parents=True)

print("Imports OK")


## 2. Logger


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("../reports/karasek/logs/karasek.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("Logger initialisé")


## 3. Labels et Ordres d'affichage

Dictionnaires de référence pour les graphiques :

- `VAR_LABELS` : libellés lisibles des variables
- `MODALITY_ORDER` : ordre d'affichage des modalités
- `CAT_VARS` : variables socio-démographiques à visualiser


In [ ]:
# ── Labels lisibles pour les graphiques ─────────────────────────────────────
VAR_LABELS: Dict[str, str] = {
    "Genre": "le genre",
    "Situation matrimonial": "la situation matrimoniale",
    "Catégorie Socio": "la catégorie socioprofessionnelle",
    "Direction": "la direction",
    "Fonction": "la fonction",
    "Poste de travail": "le poste de travail",
    "Tranche_age": "la tranche d'âge",
    "Tranche_age_binaire": "la tranche d'âge (binaire)",
    "Tranche_anciennete": "l'ancienneté",
    "Categorie_IMC": "la catégorie IMC (OMS)",
    "IMC_binaire": "l'IMC (normal / surpoids-obésité)",
    "Consommation reguliere d\u2019alcool": "la consommation régulière d'alcool",
    "tabagisme": "le tabagisme",
    "Pratique reguliere du sport": "la pratique régulière du sport",
    "Avez-vous un handicap physique": "la présence d'un handicap physique",
    "Avez-vous une maladie chronique": "la présence d'une maladie chronique",
    "Avez-vous été suivi pour un probleme psychologique *": "le suivi psychologique",
    "Dem_score_cat": "le niveau de demande psychologique",
    "Lat_score_cat": "le niveau de latitude décisionnelle",
    "SS_score_cat": "le niveau de soutien social",
    "comp_score_cat": "le niveau d'utilisation des compétences",
    "auto_score_cat": "le niveau d'autonomie décisionnelle",
    "sup_score_cat": "le niveau de soutien hiérarchique",
    "col_score_cat": "le niveau de soutien des collègues",
    "rec_score_cat": "le niveau de reconnaissance",
    "equ_score_cat": "le niveau d'équité de charge",
    "cult_score_cat": "le niveau d'adhésion à la culture",
    "adq_resources_score_cat": "le niveau d'adéquation ressources / objectifs",
    "adq_role_score_cat": "le niveau d'adéquation formation / rôle",
    "sat_score_cat": "le niveau de satisfaction au travail",
    "Karasek_quadrant": "le quadrant de Karasek (seuil théorique)",
    "Job_strain": "le Job Strain (seuil théorique)",
    "Iso_strain": "l'Isolement au Travail (seuil théorique)",
}

# ── Ordres d'affichage des modalités ────────────────────────────────────────
MODALITY_ORDER: Dict[str, List[str]] = {
    "Genre": ["Homme", "Femme"],
    "Situation matrimonial": ["Célibataire", "Marié(e)", "Divorcé(e)", "Veuf/Veuve"],
    "Tranche_age": ["20-30 ans", "31-40 ans", "41-50 ans", "51 ans et plus"],
    "Tranche_age_binaire": ["moins de 40 ans", "plus de 40 ans"],
    "Tranche_anciennete": [
        "0-2 ans",
        "3-5 ans",
        "6-10 ans",
        "11-20 ans",
        "21 ans et +",
    ],
    "Categorie_IMC": [
        "Insuffisance pondérale",
        "Corpulence normale",
        "Surpoids",
        "Obésité",
    ],
    "IMC_binaire": ["Normal", "Surpoids/Obésité"],
    "Catégorie Socio": ["Employé", "Agent de maîtrise", "Cadre"],
    "Karasek_quadrant": ["Actif", "Detendu", "Passif", "Tendu"],
    "Job_strain": ["Absent", "Présent"],
    "Iso_strain": ["Absent", "Présent"],
}
SCORE_CAT_ORDER = ["Faible", "Élevé"]

# ── Variables catégorielles à visualiser ────────────────────────────────────
CAT_VARS = [
    "Genre",
    "Situation matrimonial",
    "Tranche_age",
    "Tranche_anciennete",
    "Categorie_IMC",
    "IMC_binaire",
    "Direction",
    "Catégorie Socio",
    "Consommation reguliere d\u2019alcool",
    "tabagisme",
    "Pratique reguliere du sport",
    "Avez-vous un handicap physique",
    "Avez-vous une maladie chronique",
    "Avez-vous été suivi pour un probleme psychologique *",
]

print("Labels et ordres définis")


## 4. Définition des tableaux croisés

Listes de paires `(variable_ligne, variable_colonne)` pour les analyses croisées.


In [ ]:
CROSSTABS_RISK_CORE = [
    ("Genre", "Karasek_quadrant"),
    ("Tranche_age", "Karasek_quadrant"),
    ("Tranche_anciennete", "Karasek_quadrant"),
    ("Categorie_IMC", "Karasek_quadrant"),
    ("IMC_binaire", "Karasek_quadrant"),
    ("Avez-vous une maladie chronique", "Karasek_quadrant"),
    ("Pratique reguliere du sport", "Karasek_quadrant"),
    ("Direction", "Karasek_quadrant"),
    ("Catégorie Socio", "Karasek_quadrant"),
    ("Situation matrimonial", "Karasek_quadrant"),
]
CROSSTABS_SOCIO_RPS = [
    ("Genre", "Iso_strain"),
    ("Tranche_age", "Iso_strain"),
    ("Tranche_anciennete", "Iso_strain"),
    ("Categorie_IMC", "Iso_strain"),
    ("Avez-vous une maladie chronique", "Iso_strain"),
    ("Situation matrimonial", "Iso_strain"),
    ("Genre", "Job_strain"),
    ("Tranche_age", "Job_strain"),
    ("Tranche_anciennete", "Job_strain"),
    ("Categorie_IMC", "Job_strain"),
    ("Pratique reguliere du sport", "Job_strain"),
    ("Avez-vous une maladie chronique", "Job_strain"),
    ("Situation matrimonial", "Job_strain"),
]
CROSSTABS_CLIMATE = [
    ("sat_score_cat", "cult_score_cat"),
    ("Genre", "cult_score_cat"),
    ("Tranche_age", "cult_score_cat"),
    ("Tranche_anciennete", "cult_score_cat"),
    ("Genre", "equ_score_cat"),
    ("Tranche_age", "equ_score_cat"),
    ("Tranche_anciennete", "equ_score_cat"),
]
CROSSTABS_SOCIO_CLIMATE = [
    ("SS_score_cat", "sat_score_cat"),
    ("col_score_cat", "sat_score_cat"),
    ("sup_score_cat", "sat_score_cat"),
    ("rec_score_cat", "sat_score_cat"),
    ("equ_score_cat", "sat_score_cat"),
    ("adq_resources_score_cat", "sat_score_cat"),
    ("adq_role_score_cat", "sat_score_cat"),
    ("Genre", "rec_score_cat"),
    ("Tranche_age", "rec_score_cat"),
    ("Tranche_anciennete", "rec_score_cat"),
]
ALL_CROSSTABS = (
    CROSSTABS_RISK_CORE
    + CROSSTABS_SOCIO_RPS
    + CROSSTABS_CLIMATE
    + CROSSTABS_SOCIO_CLIMATE
)

print(f"{len(ALL_CROSSTABS)} tableaux croisés définis")


## 5. Configuration (`Config`)

Classe centrale regroupant tous les paramètres :

- chemins fichiers
- mapping de renommage des items Likert
- items à inverser
- multiplicateurs et composition des scores
- **seuils théoriques** (point central de l'échelle)
- palettes de couleurs


In [ ]:
class Config:
    # ── Chemins ─────────────────────────────────────────────────────────────
    input_file = "../lib/data/sample_karasek.csv"
    output_file = "../lib/data/sample_karasek_clean.csv"
    figures_dir = "../reports/karasek/figures"
    LIKERT_MIN = 1
    LIKERT_MAX = 4

    # ── Renommage Likert ─────────────────────────────────────────────────────
    RENAME_MAPPING: Dict[str, str] = {
        "Dans mon travail, je dois apprendre des choses nouvelles":                        "Q1_comp",
        "Dans mon travail j\u2019effectue des t\u00e2ches r\u00e9p\u00e9titives":         "Q2_comp",
        "Mon travail me demande d\u2019\u00eatre cr\u00e9atif":                            "Q3_comp",
        "Mon travail me demande un haut niveau de comp\u00e9tence":                        "Q4_comp",
        "Dans mon travail, j\u2019ai des activit\u00e9s vari\u00e9es":                    "Q5_comp",
        "J\u2019ai l\u2019occasion de d\u00e9velopper mes comp\u00e9tences professionnelles": "Q6_comp",
        "Mon travail me permet souvent de prendre des d\u00e9cisions moi-m\u00eame":       "Q1_auto",
        "Dans ma t\u00e2che, j\u2019ai tr\u00e8s peu de libert\u00e9 pour d\u00e9cider comment je fais mon travail": "Q2_auto",
        "J\u2019ai la possibilit\u00e9 d\u2019influencer le d\u00e9roulement de mon travail": "Q3_auto",
        "Mon travail me demande de travailler tr\u00e8s vite":                             "Q1_dem",
        "Mon travail demande de travailler intensement":                                    "Q2_dem",
        "On me demande d\u2019effectuer une quantit\u00e9 de travail excessive":            "Q3_dem",
        "Je dispose du temps n\u00e9cessaire pour effectuer correctement mon travail":      "Q4_dem",
        "Je re\u00e7ois des ordres contradictoires de la part d\u2019autres personnes":    "Q5_dem",
        "Mon travail n\u00e9cessite de longues p\u00e9riodes de concentration intense":    "Q6_dem",
        "Mes t\u00e2ches sont souvent interrompues avant d\u2019\u00eatre achev\u00e9es, n\u00e9cessitant de les reprendre plus tard": "Q7_dem",
        "Mon travail est \u00ab\u00a0tr\u00e8s bouscul\u00e9\u00a0\u00bb ":              "Q8_dem",
        "Attendre le travail de coll\u00e8gues ralentit souvent mon propre travail":        "Q9_dem",
        "Mon sup\u00e9rieur se sent concern\u00e9 par le bien-\u00eatre de ses subordonn\u00e9s": "Q1_sup",
        "Mon sup\u00e9rieur pr\u00eate attention \u00e0 ce que je dis":                    "Q2_sup",
        "Mon sup\u00e9rieur m\u2019aide \u00e0 mener ma t\u00e2che \u00e0 bien":           "Q3_sup",
        "Mon sup\u00e9rieur r\u00e9ussit facilement \u00e0 faire collaborer ses subordonn\u00e9s": "Q4_sup",
        "Les coll\u00e8gues avec qui je travaille sont des gens professionnellement comp\u00e9tent": "Q1_col",
        "Les coll\u00e8gues avec qui je travaille me manifestent de l\u2019int\u00e9r\u00eat": "Q2_col",
        "Les coll\u00e8gues avec qui je travaille sont amicaux":                            "Q3_col",
        "Les coll\u00e8gues avec qui je travaille m\u2019aident \u00e0 mener les t\u00e2ches \u00e0 bien": "Q4_col",
        "On me traite injustement dans mon travail":                                        "Q1_rec",
        "Ma s\u00e9curit\u00e9 d\u2019emploi est menac\u00e9e":                             "Q2_rec",
        "Ma position professionnelle actuelle correspond bien \u00e0 ma formation":         "Q3_rec",
        "Vu tous mes efforts, je re\u00e7ois le respect et l\u2019estime que je m\u00e9rite": "Q4_rec",
        "Vu tous mes efforts, mes perspectives de promotion sont satisfaisantes":            "Q5_rec",
        "Vu tous mes efforts, mon salaire est satisfaisant":                                "Q6_rec",
        "La charge de travail est r\u00e9partie \u00e9quitablement dans mon \u00e9quipe":  "Q1_equ",
        "Je m\u2019identifie \u00e0 la culture de l\u2019entreprise?":                     "Q1_cult",
        "Je m'identifie à la culture de l'entreprise?":                                     "Q1_cult",
        "Je recommanderai ma compagnie \u00e0 mes connaissances \u00e0 la recherche d\u2019un emploi": "Q2_cult",
        "Je recommanderai ma compagnie à mes connaissances à la recherche d'un emploi":     "Q2_cult",
        "Je suis satisfait de mon travail dans la compagnie":                               "Q1_sat",
        "Je sais ce que je dois faire pour atteindre les objectifs qui me sont fix\u00e9s": "Q1_adq_resources",
        "Je dispose de toutes les ressources nécessaires à l'accomplissement de mes taches quotidiennes": "Q2_adq_resources",
        "Mes besoins de formations sont bien pris en compte":                               "Q1_adq_role",
        "Les formations dispensées sont cohérentes avec les taches dont j'ai la responsabilité ou qui me sont assignées": "Q2_adq_role",
    }

    INVERT_ITEMS: List[str] = ["Q2_auto", "Q2_comp", "Q4_dem", "Q1_rec", "Q2_rec"]

    SCORE_MULTIPLIERS: Dict[str, int] = {
        "comp": 2,
        "auto": 4,
        "dem": 1,
        "sup": 1,
        "col": 1,
    }
    KARASEK_SCORE_COMPOSITION: Dict[str, List[str]] = {
        "Dem_score": ["dem_score"],
        "Lat_score": ["comp_score", "auto_score"],
        "SS_score": ["sup_score", "col_score"],
    }
    RH_SCORE_GROUPS: List[str] = [
        "rec",
        "equ",
        "cult",
        "adq_resources",
        "adq_role",
        "sat",
    ]
    BINARY_CATEGORY_LABELS = ("Faible", "Élevé")

    # ── Seuils théoriques (dashboard) ───────────────────────────────────────
    # Principe : seuil = point médian de l'échelle Likert (1–4) = 2.5 par item
    #   Dem_score  = 9  items × 2.5 = 22.5
    #   comp_score = 6  items × 2   × 2.5 = 30.0
    #   auto_score = 3  items × 4   × 2.5 = 30.0
    #   Lat_score  = comp_score + auto_score = 60.0
    #   sup_score  = 4  items × 2.5 = 10.0
    #   col_score  = 4  items × 2.5 = 10.0
    #   SS_score   = sup_score + col_score  = 20.0
    DEM: float = 22.5
    LAT: float = 60.0
    SS: float = 20.0

    RH_THRESHOLDS: Dict[str, float] = {
        "rec_score": 15.0,
        "equ_score": 2.5,
        "cult_score": 5.0,
        "adq_resources_score": 5.0,
        "adq_role_score": 5.0,
        "sat_score": 2.5,
    }
    KARASEK_PALETTE = {
        "Actif": "#2ca02c",
        "Detendu": "#1f77b4",
        "Tendu": "#d62728",
        "Passif": "#7f7f7f",
    }
    STRAIN_PALETTE = {"Présent": "#d62728", "Absent": "#2ca02c"}
    SCORE_LABELS: Dict[str, str] = {
        "Dem_score": "Demande psychologique",
        "Lat_score": "Latitude décisionnelle",
        "SS_score": "Soutien social",
        "comp_score": "Utilisation des compétences",
        "auto_score": "Autonomie décisionnelle",
        "sup_score": "Soutien hiérarchique",
        "col_score": "Soutien des collègues",
        "rec_score": "Reconnaissance",
        "equ_score": "Équité de charge",
        "cult_score": "Culture organisationnelle",
        "adq_resources_score": "Adéquation ressources / objectifs",
        "adq_role_score": "Adéquation besoins en formation / rôle",
        "sat_score": "Satisfaction au travail",
    }
    COMPANY_NAME = "COMPANY NAME"


config = Config()
print("Config instanciée")
print(f"   Seuils théoriques → DEM={config.DEM}  LAT={config.LAT}  SS={config.SS}")


## 6. Nettoyage des données (`Cleaner`)

Fonctions de nettoyage :

- `clean_likert` : validation des valeurs Likert (1–4)
- `invert_items` : inversion des items à cotation inversée
- `compute_group_score` : calcul des scores composites
- `enrich_sociodem` : création des variables dérivées (tranches d'âge, IMC, ancienneté)


In [ ]:
def imc_category(imc) -> str:
    if pd.isna(imc):
        return "Manquant"
    if imc < 18.5:
        return "Insuffisance pondérale"
    if imc < 25:
        return "Corpulence normale"
    if imc < 30:
        return "Surpoids"
    return "Obésité"


class Cleaner:
    @staticmethod
    def clean_likert(
        series: pd.Series,
        min_val: int = Config.LIKERT_MIN,
        max_val: int = Config.LIKERT_MAX,
    ) -> pd.Series:
        s = pd.to_numeric(series, errors="coerce")
        invalid = (~s.between(min_val, max_val)) & s.notna()
        if invalid.any():
            logger.warning(
                f"{invalid.sum()} valeurs hors plage [{min_val},{max_val}] "
                f"→ NaN  ('{series.name}')"
            )
        return s.where(s.between(min_val, max_val))

    @staticmethod
    def invert_items(
        df: pd.DataFrame,
        items: List[str],
        min_val: int = Config.LIKERT_MIN,
        max_val: int = Config.LIKERT_MAX,
    ) -> pd.DataFrame:
        df = df.copy()
        available = [c for c in items if c in df.columns]
        missing = set(items) - set(available)
        if missing:
            logger.info(f"Items à inverser absents : {missing}")
        if available:
            df[available] = (min_val + max_val) - df[available]
            logger.info(f"Inversion appliquée : {available}")
        return df

    @staticmethod
    def compute_group_score(
        df: pd.DataFrame, suffix: str, multiplier: int = 1, normalize: bool = True
    ) -> pd.Series:
        cols = [c for c in df.columns if c.endswith(f"_{suffix}")]
        if not cols:
            logger.warning(f"Aucun item pour le suffixe '_{suffix}'")
            return pd.Series(np.nan, index=df.index, name=f"{suffix}_score")
        ssum = df[cols].sum(axis=1, skipna=True)
        n_answered = df[cols].notna().sum(axis=1)
        n_items = len(cols)
        with np.errstate(invalid="ignore", divide="ignore"):
            score = (
                ssum / n_answered.replace(0, np.nan) * n_items * multiplier
                if normalize
                else ssum * multiplier
            )
        score = score.where(n_answered > 0, np.nan)
        score.name = f"{suffix}_score"
        return score

    @staticmethod
    def enrich_sociodem(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if "Age" in df.columns:
            age_num = pd.to_numeric(df["Age"], errors="coerce")
            df["Tranche_age"] = (
                pd.cut(
                    age_num,
                    bins=[0, 30, 40, 50, np.inf],
                    labels=["20-30 ans", "31-40 ans", "41-50 ans", "51 ans et plus"],
                    right=True,
                )
                .astype(str)
                .replace("nan", np.nan)
            )
            df["Tranche_age"] = df["Tranche_age"].where(
                df["Tranche_age"] != "nan", np.nan
            )
            df["Tranche_age_binaire"] = np.where(
                age_num <= 40,
                "moins de 40 ans",
                np.where(age_num.isna(), np.nan, "plus de 40 ans"),
            )
        anc_col = "Anciennet\u00e9"
        if anc_col in df.columns:
            anc_num = pd.to_numeric(df[anc_col], errors="coerce")
            df["Tranche_anciennete"] = pd.cut(
                anc_num,
                bins=[-1, 2, 5, 10, 20, np.inf],
                labels=["0-2 ans", "3-5 ans", "6-10 ans", "11-20 ans", "21 ans et +"],
            )
        if "Poids" in df.columns and "Taille" in df.columns:
            poids = pd.to_numeric(df["Poids"], errors="coerce")
            taille = pd.to_numeric(df["Taille"], errors="coerce")
            taille_m = taille / 100 if taille.median() > 3 else taille
            imc_num = poids / (taille_m**2)
            df["Categorie_IMC"] = imc_num.apply(imc_category)
            df["IMC_binaire"] = np.where(
                df["Categorie_IMC"].isin(
                    ["Insuffisance pondérale", "Corpulence normale"]
                ),
                "Normal",
                np.where(df["Categorie_IMC"] == "Manquant", np.nan, "Surpoids/Obésité"),
            )
        return df


cleaner = Cleaner()
print("Cleaner défini")


## 7. Classification Karasek (`KarasekClassifier`)

- **`classify_by_thresholds`** : classification par seuil théorique → 4 quadrants + Job Strain + Iso Strain
- **`categorize_scores_by_threshold`** : catégorisation binaire (Faible / Élevé) de tous les scores


In [ ]:
class KarasekClassifier:
    @staticmethod
    def classify_by_thresholds(
        df: pd.DataFrame,
        dem_threshold: float,
        lat_threshold: float,
        ss_threshold: float,
    ) -> pd.DataFrame:
        """
        Classification par seuil théorique (dashboard / monitoring inter-vagues).
        Produit : Karasek_quadrant, Job_strain, Iso_strain
        Libellé : 'niveau supérieur au point central de l'échelle' — pas 'à risque'
        """
        df = df.copy()
        missing = [
            c for c in ["Dem_score", "Lat_score", "SS_score"] if c not in df.columns
        ]
        if missing:
            raise ValueError(f"Scores manquants : {missing}")

        logger.info(
            f"[THÉORIQUE] Dem:{dem_threshold}  Lat:{lat_threshold}  SS:{ss_threshold}"
        )

        df["Karasek_quadrant"] = np.select(
            [
                (df["Lat_score"] >= lat_threshold) & (df["Dem_score"] >= dem_threshold),
                (df["Lat_score"] >= lat_threshold) & (df["Dem_score"] < dem_threshold),
                (df["Lat_score"] < lat_threshold) & (df["Dem_score"] >= dem_threshold),
            ],
            ["Actif", "Detendu", "Tendu"],
            default="Passif",
        )
        df["Job_strain"] = np.where(
            (df["Dem_score"] >= dem_threshold) & (df["Lat_score"] < lat_threshold),
            "Présent",
            "Absent",
        )
        df["Iso_strain"] = np.where(
            (df["Dem_score"] >= dem_threshold) & (df["SS_score"] < ss_threshold),
            "Présent",
            "Absent",
        )
        df.attrs["thresholds"] = {
            "method": "theoretical_midpoint",
            "usage": "dashboard_monitoring",
            "note": "Seuil basé sur le point médian de l'échelle, indicateur conventionnel non clinique",
            "Dem_threshold": float(dem_threshold),
            "Lat_threshold": float(lat_threshold),
            "SS_threshold": float(ss_threshold),
        }
        logger.info("[THÉORIQUE] Classification terminée")
        return df

    @staticmethod
    def categorize_scores_by_threshold(
        df: pd.DataFrame,
        score_thresholds: Dict[str, float],
    ) -> pd.DataFrame:
        """
        Catégorisation binaire par seuil théorique → colonnes *_cat (Faible / Élevé).
        Les scores continus sont toujours conservés.
        """
        df = df.copy()
        labels = Config.BINARY_CATEGORY_LABELS
        for col, threshold in score_thresholds.items():
            if col not in df.columns:
                logger.warning(f"Score absent, ignoré : {col}")
                continue
            cat_col = f"{col}_cat"
            df[cat_col] = np.where(
                df[col].isna(),
                "Non renseigné",
                np.where(df[col] <= threshold, labels[0], labels[1]),
            )
        return df


classifier = KarasekClassifier()
print("Classifier défini")


## 8. Utilitaires de tri des modalités


In [ ]:
def _sort_modalities(series: pd.Series, col_name: str) -> List:
    present = list(series.dropna().unique())
    if col_name in MODALITY_ORDER and MODALITY_ORDER[col_name]:
        ordered = [m for m in MODALITY_ORDER[col_name] if m in present]
        ordered += [m for m in present if m not in ordered]
        return ordered
    if set(SCORE_CAT_ORDER).intersection(present):
        ordered = [m for m in SCORE_CAT_ORDER if m in present]
        ordered += [m for m in present if m not in ordered]
        return ordered
    return sorted(present, key=str)


print("Utilitaires définis")


## 9. Visualisations (`KarasekPlotter`)

Classe produisant l'ensemble des graphiques :

- `plot_categorical` : barplots simples socio-démographiques
- `plot_stacked_bar` : barplots empilés (tableaux croisés visuels)
- `plot_karasek_scatter` : nuage de points Demande × Latitude
- `plot_age_pyramid` : pyramide des âges
- `plot_karasek_direction_heatmap` : heatmap quadrants par direction


In [ ]:
class KarasekPlotter:
    TITLE_SIZE = 15
    LABEL_SIZE = 12
    TICK_SIZE = 10
    ANNOT_SIZE = 13
    LEGEND_SIZE = 10
    TITLE_WRAP = 60
    YLABEL_WRAP = 20
    LEGEND_PAD = 1.05
    RIGHT_MARGIN = 0.78
    STACKED_ANNOT_SIZE = 13
    STACKED_ANNOT_COLOR = "white"
    STACKED_ANNOT_WEIGHT = "bold"
    PYRAMID_COLOR_HOMME = "#4472C4"
    PYRAMID_COLOR_FEMME = "#FF69B4"
    PYRAMID_ANNOT_SEUIL = 0.18
    LOWER_WORDS = {"le", "la", "les", "de", "du", "des", "au", "aux"}
    KARASEK_PALETTE = Config.KARASEK_PALETTE
    STRAIN_PALETTE = Config.STRAIN_PALETTE
    SCORE_CAT_PALETTE = {"Faible": "#d62728", "Élevé": "#2ca02c"}
    INV_SCORE_CAT_PALETTE = {"Faible": "#2ca02c", "Élevé": "#d62728"}
    MUTED = "#7f7f7f"
    HEATMAP_VMIN = 0
    HEATMAP_VMAX = 60

    def __init__(self, config):
        self.config = config
        self.base_dir = Path(config.figures_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _wrap(text: str, width: int) -> str:
        return "\n".join(textwrap.wrap(text, width=width))

    @staticmethod
    def _norm(text: str) -> str:
        return (
            unicodedata.normalize("NFD", str(text).lower().strip())
            .encode("ascii", "ignore")
            .decode()
        )

    @classmethod
    def _smart_capitalize(cls, text: str) -> str:
        words = text.split()
        result = []
        for i, w in enumerate(words):
            lw = w.lower()
            if i == 0:
                result.append(w.capitalize())
            elif lw in cls.LOWER_WORDS or lw.startswith("l'") or lw.startswith("d'"):
                result.append(lw)
            else:
                result.append(w.capitalize())
        return " ".join(result)

    def _get_palette(self, categories: List, col: str) -> Dict:
        n = self._norm(col)
        if "karasek" in n or "quadrant" in n:
            return {c: self.KARASEK_PALETTE.get(c, self.MUTED) for c in categories}
        if "strain" in n:
            return {c: self.STRAIN_PALETTE.get(c, self.MUTED) for c in categories}
        if "score" in n or any(c in SCORE_CAT_ORDER for c in categories):
            if col.startswith("Dem_") or "dem_" in col.lower():
                return {
                    c: self.INV_SCORE_CAT_PALETTE.get(c, self.MUTED) for c in categories
                }
            return {c: self.SCORE_CAT_PALETTE.get(c, self.MUTED) for c in categories}
        colors = sns.color_palette("colorblind", len(categories))
        return dict(zip(categories, colors))

    def _save(self, folder: str, filename: str) -> str:
        d = self.base_dir / folder
        d.mkdir(parents=True, exist_ok=True)
        fp = d / filename
        plt.savefig(fp, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()
        return str(fp)

    @staticmethod
    def _fname(col: str) -> str:
        return "".join(c if c.isalnum() or c in "_-" else "_" for c in col)[:80]

    @staticmethod
    def _label(col: str) -> str:
        return VAR_LABELS.get(col, col)

    # ── Barplot simple ───────────────────────────────────────────────────────
    def plot_categorical(self, df: pd.DataFrame, col: str) -> Optional[str]:
        if col not in df.columns:
            return None
        data = df[col].dropna()
        if data.empty:
            return None
        mods = _sort_modalities(data, col)
        counts = data.value_counts().reindex(mods).fillna(0).astype(int)
        pcts = (counts / counts.sum() * 100).round(1)
        colors = [self._get_palette(mods, col).get(m, self.MUTED) for m in mods]
        fig_h = max(4, len(mods) * 0.65 + 1.5)
        fig, ax = plt.subplots(figsize=(9, fig_h), constrained_layout=True)
        bars = ax.barh(mods, pcts, color=colors, edgecolor="white", height=0.6)
        for bar, pct, n in zip(bars, pcts, counts):
            if pct > 0:
                ax.text(
                    bar.get_width() + 0.8,
                    bar.get_y() + bar.get_height() / 2,
                    f"{pct}%  ({n})",
                    va="center",
                    ha="left",
                    fontsize=self.ANNOT_SIZE,
                    fontweight="bold",
                )
        ax.set_xlim(0, 120)
        col_label = self._smart_capitalize(self._label(col))
        ax.set_title(
            self._wrap(
                f"Répartition des employés de {self.config.COMPANY_NAME} selon {col_label}",
                self.TITLE_WRAP,
            ),
            fontsize=self.TITLE_SIZE,
            pad=14,
        )
        ax.set_xlabel("Pourcentage (%)", fontsize=self.LABEL_SIZE)
        ax.set_ylabel(
            self._wrap(col_label, self.YLABEL_WRAP),
            fontsize=self.LABEL_SIZE,
            labelpad=10,
        )
        ax.tick_params(axis="both", labelsize=self.TICK_SIZE)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        return self._save("sociodem", f"{self._fname(col)}_barplot.png")

    # ── Stacked bar ──────────────────────────────────────────────────────────
    def plot_stacked_bar(
        self, df: pd.DataFrame, x_col: str, hue_col: str
    ) -> Optional[str]:
        if x_col not in df.columns or hue_col not in df.columns:
            return None
        data = df[[x_col, hue_col]].dropna()
        if data.empty:
            return None
        x_order = _sort_modalities(data[x_col], x_col)
        hue_order = _sort_modalities(data[hue_col], hue_col)
        ct = pd.crosstab(data[x_col], data[hue_col])
        ct = ct.reindex(index=x_order, columns=hue_order, fill_value=0)
        pct = ct.div(ct.sum(axis=1), axis=0) * 100
        palette = self._get_palette(hue_order, hue_col)
        colors = [palette.get(m, self.MUTED) for m in hue_order]
        fig_h = max(4, len(x_order) * 0.8 + 2.5)
        fig, ax = plt.subplots(figsize=(13, fig_h))
        left = np.zeros(len(x_order))
        for i, hv in enumerate(hue_order):
            widths = pct[hv].values
            counts = ct[hv].values
            bars = ax.barh(
                x_order,
                widths,
                left=left,
                color=colors[i],
                edgecolor="white",
                label=self._smart_capitalize(str(hv)),
                height=0.65,
            )
            for bar, w, n in zip(bars, widths, counts):
                if w > 0:
                    x_c = bar.get_x() + bar.get_width() / 2
                    y_c = bar.get_y() + bar.get_height() / 2
                    if w < 6:
                        ax.text(
                            bar.get_x() + bar.get_width() + 0.5,
                            y_c,
                            f"{w:.1f}%\n({n})",
                            ha="left",
                            va="center",
                            fontsize=self.STACKED_ANNOT_SIZE,
                            fontweight=self.STACKED_ANNOT_WEIGHT,
                            color="#333333",
                        )
                    else:
                        ax.text(
                            x_c,
                            y_c,
                            f"{w:.1f}%\n({n})",
                            ha="center",
                            va="center",
                            fontsize=self.STACKED_ANNOT_SIZE,
                            fontweight=self.STACKED_ANNOT_WEIGHT,
                            color=self.STACKED_ANNOT_COLOR,
                        )
            left += widths
        x_label = self._smart_capitalize(self._label(x_col))
        hue_label = self._smart_capitalize(self._label(hue_col))
        ax.set_xlim(0, 100)
        ax.set_title(
            self._wrap(
                f"Répartition des employés de {self.config.COMPANY_NAME} "
                f"selon {x_label} et {hue_label}",
                self.TITLE_WRAP,
            ),
            fontsize=self.TITLE_SIZE,
            pad=14,
        )
        ax.set_xlabel("Pourcentage (%)", fontsize=self.LABEL_SIZE)
        ax.set_ylabel(
            self._wrap(x_label, self.YLABEL_WRAP), fontsize=self.LABEL_SIZE, labelpad=10
        )
        ax.tick_params(axis="both", labelsize=self.TICK_SIZE)
        legend = ax.legend(
            title=hue_label,
            fontsize=self.LEGEND_SIZE,
            title_fontsize=self.LEGEND_SIZE,
            bbox_to_anchor=(self.LEGEND_PAD, 1),
            loc="upper left",
            frameon=True,
        )
        legend.get_title().set_fontweight("bold")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.subplots_adjust(right=self.RIGHT_MARGIN, top=0.88, bottom=0.10, left=0.18)
        fname = f"{self._fname(hue_col)}_by_{self._fname(x_col)}.png"
        return self._save("stacked", fname)

    # ── Scatterplot Karasek ──────────────────────────────────────────────────
    def plot_karasek_scatter(
        self,
        df: pd.DataFrame,
        figsize: tuple = (10, 8),
        point_size: int = 60,
        alpha: float = 0.75,
    ) -> Optional[str]:
        quad_col = "Karasek_quadrant"
        required = ["Dem_score", "Lat_score", quad_col]
        if any(c not in df.columns for c in required):
            return None
        data = df[required].dropna()
        if data.empty:
            return None
        dem_line = self.config.DEM
        lat_line = self.config.LAT
        QUAD_ORDER = ["Actif", "Detendu", "Tendu", "Passif"]
        present_quads = [q for q in QUAD_ORDER if q in data[quad_col].unique()]
        x_min, x_max = data["Dem_score"].min(), data["Dem_score"].max()
        y_min, y_max = data["Lat_score"].min(), data["Lat_score"].max()
        mx = (x_max - x_min) * 0.05
        my = (y_max - y_min) * 0.05
        x_lo, x_hi = x_min - mx, x_max + mx
        y_lo, y_hi = y_min - my, y_max + my
        QPAL = self.KARASEK_PALETTE
        fig, ax = plt.subplots(figsize=figsize)
        za = 0.06
        ax.fill_between([dem_line, x_hi], lat_line, y_hi, color=QPAL["Actif"], alpha=za)
        ax.fill_between(
            [x_lo, dem_line], lat_line, y_hi, color=QPAL["Detendu"], alpha=za
        )
        ax.fill_between([dem_line, x_hi], y_lo, lat_line, color=QPAL["Tendu"], alpha=za)
        ax.fill_between(
            [x_lo, dem_line], y_lo, lat_line, color=QPAL["Passif"], alpha=za
        )
        ax.axvline(dem_line, color="black", linestyle="--", linewidth=1.2)
        ax.axhline(lat_line, color="black", linestyle=":", linewidth=1.2)
        for quad in present_quads:
            subset = data[data[quad_col] == quad]
            pct = len(subset) / len(data) * 100
            ax.scatter(
                subset["Dem_score"],
                subset["Lat_score"],
                c=QPAL[quad],
                label=f"{quad} ({pct:.1f}%)",
                s=point_size,
                alpha=alpha,
                edgecolors="white",
                linewidths=0.4,
            )
        for quad, (qx, qy, ha, va) in {
            "Actif": (x_hi - mx * 0.5, y_hi - my * 0.5, "right", "top"),
            "Detendu": (x_lo + mx * 0.5, y_hi - my * 0.5, "left", "top"),
            "Tendu": (x_hi - mx * 0.5, y_lo + my * 0.5, "right", "bottom"),
            "Passif": (x_lo + mx * 0.5, y_lo + my * 0.5, "left", "bottom"),
        }.items():
            if quad in present_quads:
                ax.text(
                    qx,
                    qy,
                    quad.upper(),
                    color=QPAL[quad],
                    fontsize=9,
                    fontweight="bold",
                    ha=ha,
                    va=va,
                    alpha=0.55,
                    zorder=5,
                )
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(y_lo, y_hi)
        ax.set_title(
            self._wrap(
                f"Répartition des employés de {self.config.COMPANY_NAME} — "
                "Demande Psychologique × Latitude Décisionnelle\n(Seuil théorique — point central de l'échelle)",
                self.TITLE_WRAP,
            ),
            fontsize=self.TITLE_SIZE,
            pad=14,
        )
        ax.set_xlabel("Demande Psychologique", fontsize=self.LABEL_SIZE)
        ax.set_ylabel(
            self._wrap("Latitude Décisionnelle", self.YLABEL_WRAP),
            fontsize=self.LABEL_SIZE,
            labelpad=10,
        )
        ax.tick_params(axis="both", labelsize=self.TICK_SIZE)
        legend = ax.legend(
            title="Quadrant de Karasek",
            bbox_to_anchor=(self.LEGEND_PAD, 1),
            loc="upper left",
            fontsize=self.LEGEND_SIZE,
            title_fontsize=self.LEGEND_SIZE,
            frameon=True,
        )
        legend.get_title().set_fontweight("bold")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.subplots_adjust(right=self.RIGHT_MARGIN, top=0.88, bottom=0.10, left=0.12)
        return self._save("scatter", "karasek_scatter.png")

    # ── Pyramide des âges ────────────────────────────────────────────────────
    def plot_age_pyramid(self, df: pd.DataFrame) -> Optional[str]:
        if "Tranche_age" not in df.columns or "Genre" not in df.columns:
            return None
        data = df[["Tranche_age", "Genre"]].dropna()
        if data.empty:
            return None
        age_order = _sort_modalities(data["Tranche_age"], "Tranche_age")
        ct = pd.crosstab(data["Tranche_age"], data["Genre"]).reindex(
            age_order, fill_value=0
        )
        totaux = ct.sum(axis=0)
        ct_pct = ct.div(totaux, axis=1) * 100
        fig, ax = plt.subplots(figsize=(9, 7))
        homme_vals = ct_pct.get("Homme", pd.Series(0, index=age_order))
        femme_vals = ct_pct.get("Femme", pd.Series(0, index=age_order))
        homme_n = ct.get("Homme", pd.Series(0, index=age_order))
        femme_n = ct.get("Femme", pd.Series(0, index=age_order))
        max_val = max(homme_vals.max(), femme_vals.max()) * 1.15
        seuil = max_val * self.PYRAMID_ANNOT_SEUIL
        bars_h = ax.barh(
            age_order, -homme_vals, label="Homme", color=self.PYRAMID_COLOR_HOMME
        )
        bars_f = ax.barh(
            age_order, femme_vals, label="Femme", color=self.PYRAMID_COLOR_FEMME
        )

        def _annotate(bars, vals, ns, side):
            for bar, pct, n in zip(bars, vals, ns):
                if pct <= 0:
                    continue
                bar_w = abs(bar.get_width())
                y_c = bar.get_y() + bar.get_height() / 2
                label = f"{pct:.1f}%\n({n})"
                if bar_w >= seuil:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        y_c,
                        label,
                        ha="center",
                        va="center",
                        fontsize=self.ANNOT_SIZE,
                        fontweight="bold",
                        color="white",
                    )
                else:
                    x_pos = (
                        bar.get_x() - 0.5
                        if side == "left"
                        else bar.get_x() + bar.get_width() + 0.5
                    )
                    ax.text(
                        x_pos,
                        y_c,
                        label,
                        ha="right" if side == "left" else "left",
                        va="center",
                        fontsize=self.ANNOT_SIZE,
                        fontweight="bold",
                        color="#333333",
                    )

        _annotate(bars_h, homme_vals, homme_n, "left")
        _annotate(bars_f, femme_vals, femme_n, "right")
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlim(-max_val, max_val)
        ticks = ax.get_xticks()
        ax.set_xticklabels([f"{abs(t):.0f}%" for t in ticks], fontsize=self.TICK_SIZE)
        ax.set_title(
            self._wrap(
                f"Répartition des employés de {self.config.COMPANY_NAME} "
                "selon la Tranche d'âge et le Genre",
                self.TITLE_WRAP,
            ),
            fontsize=self.TITLE_SIZE,
            pad=14,
        )
        ax.set_xlabel(
            "Pourcentage (%) au sein de chaque genre", fontsize=self.LABEL_SIZE
        )
        ax.set_ylabel(
            self._wrap("Tranche d'âge", self.YLABEL_WRAP),
            fontsize=self.LABEL_SIZE,
            labelpad=10,
        )
        ax.tick_params(axis="y", labelsize=self.TICK_SIZE)
        ax.legend(
            bbox_to_anchor=(self.LEGEND_PAD, 1),
            loc="upper left",
            fontsize=self.LEGEND_SIZE,
            frameon=True,
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.subplots_adjust(right=self.RIGHT_MARGIN, top=0.88, bottom=0.12, left=0.18)
        return self._save("sociodem", "age_pyramid.png")

    # ── Heatmap Karasek par direction ────────────────────────────────────────
    def plot_karasek_direction_heatmap(
        self, df: pd.DataFrame, group_col: str = "Direction"
    ) -> Optional[str]:
        quad_col = "Karasek_quadrant"
        if quad_col not in df.columns or group_col not in df.columns:
            return None
        ct = pd.crosstab(df[group_col], df[quad_col], normalize="index") * 100
        quadrants = ["Tendu", "Actif", "Passif", "Detendu"]
        for q in quadrants:
            if q not in ct.columns:
                ct[q] = 0
        ct = ct.sort_values("Tendu", ascending=True)
        colors_map = {
            "Tendu": ["#ffffff", "#e74c3c"],
            "Actif": ["#ffffff", "#3498db"],
            "Passif": ["#ffffff", "#f39c12"],
            "Detendu": ["#ffffff", "#2ecc71"],
        }
        fig, axes = plt.subplots(1, 4, figsize=(20, 1 + len(ct) * 0.4), sharey=True)
        fig.patch.set_facecolor("#f8f9fa")
        for ax, quad in zip(axes, quadrants):
            cmap = LinearSegmentedColormap.from_list(f"cmap_{quad}", colors_map[quad])
            sns.heatmap(
                ct[[quad]],
                ax=ax,
                cmap=cmap,
                vmin=self.HEATMAP_VMIN,
                vmax=self.HEATMAP_VMAX,
                annot=True,
                fmt=".0f",
                annot_kws={"size": 8},
                linewidths=0.5,
                linecolor="#eee",
                cbar=False,
            )
            ax.set_title(
                quad.upper(), fontsize=11, fontweight="bold", color=colors_map[quad][1]
            )
            ax.set_xlabel("%", fontsize=9)
            ax.tick_params(axis="y", labelsize=9)
            for spine in ax.spines.values():
                spine.set_visible(False)
        plt.suptitle(
            self._wrap(
                f"Répartition Karasek par {group_col} — seuil théorique "
                "(trié par tension décroissante)",
                80,
            ),
            fontsize=self.TITLE_SIZE,
            fontweight="bold",
            y=1.02,
        )
        return self._save("heatmaps", f"karasek_heatmap_by_{group_col}.png")


plotter = KarasekPlotter(config)
print("KarasekPlotter défini")


## 10. Tableaux croisés (`CrossTabGenerator`)

Génère et exporte les tableaux croisés vers un fichier Excel.


In [ ]:
class CrossTabGenerator:
    def __init__(self):
        Path("../reports/karasek/tables").mkdir(exist_ok=True, parents=True)

    def _build(
        self, df: pd.DataFrame, row_var: str, col_var: str, by: str = "row", rd: int = 1
    ) -> pd.DataFrame:
        tmp = df[[row_var, col_var]].dropna()
        ct = pd.crosstab(tmp[row_var], tmp[col_var])
        row_totals = ct.sum(axis=1)
        col_totals = ct.sum(axis=0)
        grand_total = ct.sum().sum()
        if by == "row":
            ct_pct = ct.div(row_totals, axis=0) * 100
            total_col_pct = pd.Series(100.0, index=ct.index)
            total_row_pct = (
                (col_totals / grand_total * 100)
                if grand_total > 0
                else pd.Series(0.0, index=col_totals.index)
            )
        else:
            ct_pct = ct.div(col_totals, axis=1) * 100
            total_row_pct = pd.Series(100.0, index=ct.columns)
            total_col_pct = (
                (row_totals / grand_total * 100)
                if grand_total > 0
                else pd.Series(0.0, index=row_totals.index)
            )
        result = pd.DataFrame(index=ct.index, columns=ct.columns, dtype=object)
        for col in ct.columns:
            for idx in ct.index:
                n = int(ct.at[idx, col])
                pct = ct_pct.at[idx, col]
                result.at[idx, col] = f"{n} ({pct:.{rd}f}%)"
        result["Total"] = [
            f"{int(row_totals[idx])} ({total_col_pct[idx]:.{rd}f}%)" for idx in ct.index
        ]
        total_row = {
            col: f"{int(col_totals[col])} ({total_row_pct[col]:.{rd}f}%)"
            for col in ct.columns
        }
        total_row["Total"] = f"{int(grand_total)} (100.0%)"
        result.loc["Total"] = pd.Series(total_row)
        return result

    def export_crosstabs(
        self,
        df: pd.DataFrame,
        crosstab_list: List[Tuple[str, str]],
        filename: str = "../reports/karasek/tables/tableaux_croises.xlsx",
    ) -> str:
        from openpyxl import Workbook
        from openpyxl.styles import Alignment, Border, Font, Side
        from openpyxl.utils.dataframe import dataframe_to_rows

        wb = Workbook()
        wb.remove(wb.active)
        thin = Border(
            **{s: Side(style="thin") for s in ("left", "right", "top", "bottom")}
        )
        ws = wb.create_sheet("Analyse_Karasek")
        ws.append(["Tableaux croisés – Analyse RPS COMPANY NAME"])
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=10)
        ws["A1"].font = Font(bold=True, size=14)
        ws["A1"].alignment = Alignment(horizontal="center")
        ws.append(["Note : colonnes * = seuil théorique (point central — dashboard)."])
        ws["A2"].font = Font(italic=True, size=10)
        cur = 4
        for rv, cv in crosstab_list:
            if rv not in df.columns or cv not in df.columns:
                continue
            for by, suffix in [("row", "% par ligne"), ("col", "% par colonne")]:
                tbl = self._build(df, rv, cv, by=by)
                ws.cell(row=cur, column=1, value=f"{rv} × {cv}  ({suffix})")
                ws.cell(row=cur, column=1).font = Font(bold=True)
                cur += 1
                for r in dataframe_to_rows(tbl, index=True, header=True):
                    ws.append(r)
                    cur += 1
                cur += 1
        for row in ws.iter_rows(
            min_row=4, max_row=cur - 1, min_col=1, max_col=ws.max_column
        ):
            for cell in row:
                cell.border = thin
                cell.alignment = Alignment(horizontal="center", vertical="center")
        Path(filename).parent.mkdir(exist_ok=True, parents=True)
        wb.save(filename)
        logger.info(f"Tableaux croisés → {filename}")
        return filename


crosstab_gen = CrossTabGenerator()
print("CrossTabGenerator défini")


## 11. Pipeline complet (`KarasekPipeline`)

Orchestre l'ensemble des étapes : chargement, nettoyage, calcul des scores, classification, export.


In [ ]:
class KarasekPipeline:
    def __init__(self, config: Config = None):
        self.config = config or Config()
        self.cleaner = Cleaner()
        self.classifier = KarasekClassifier()
        self.plotter = KarasekPlotter(self.config)
        self.crosstab_gen = CrossTabGenerator()
        self.score_cols = [
            "Dem_score",
            "Lat_score",
            "SS_score",
            "comp_score",
            "auto_score",
            "sup_score",
            "col_score",
            "rec_score",
            "equ_score",
            "cult_score",
            "adq_resources_score",
            "adq_role_score",
            "sat_score",
        ]
        self.crosstab_list = ALL_CROSSTABS

    def _rename(self, df):
        mapping = {
            k: v for k, v in self.config.RENAME_MAPPING.items() if k in df.columns
        }
        if mapping:
            df = df.rename(columns=mapping)
        return self.cleaner.enrich_sociodem(df)

    def _clean_likert(self, df):
        df = df.copy()
        lk = [c for c in df.columns if c.startswith("Q") and "_" in c]
        for col in lk:
            df[col] = self.cleaner.clean_likert(df[col])
        return df

    def _compute_scores(self, df):
        df = df.copy()
        for g, mult in self.config.SCORE_MULTIPLIERS.items():
            df[f"{g}_score"] = self.cleaner.compute_group_score(
                df, g, multiplier=mult, normalize=True
            )
        for name, comps in self.config.KARASEK_SCORE_COMPOSITION.items():
            existing = [c for c in comps if c in df.columns]
            df[name] = sum(df[c] for c in existing) if existing else np.nan
        for g in self.config.RH_SCORE_GROUPS:
            df[f"{g}_score"] = self.cleaner.compute_group_score(
                df, g, multiplier=1, normalize=True
            )
        return df

    def run(self, input_file=None, output_file=None, drop_missing=True):
        input_file = input_file or self.config.input_file
        output_file = output_file or self.config.output_file
        try:
            df_raw = pd.read_csv(
                input_file, encoding="utf-8-sig", sep=None, engine="python"
            )
        except Exception:
            df_raw = pd.read_csv(
                input_file, encoding="latin-1", sep=None, engine="python"
            )
        logger.info(f"Données chargées : {df_raw.shape}")
        df = self._rename(df_raw)
        df = self._clean_likert(df)
        df = self.cleaner.invert_items(df, self.config.INVERT_ITEMS)
        lk = [c for c in df.columns if c.startswith("Q") and "_" in c]
        if drop_missing and lk:
            before = len(df)
            df = df.dropna(subset=lk)
            logger.info(f"Lignes supprimées (NA Likert) : {before - len(df)}")
        df = self._compute_scores(df)
        # Classification théorique (seuil point central)
        df = self.classifier.classify_by_thresholds(
            df, self.config.DEM, self.config.LAT, self.config.SS
        )
        # Catégorisation binaire
        all_thresholds = {
            "Dem_score": self.config.DEM,
            "Lat_score": self.config.LAT,
            "SS_score": self.config.SS,
            **{
                sc: v for sc, v in self.config.RH_THRESHOLDS.items() if sc in df.columns
            },
            **{
                k: v
                for k, v in {
                    "comp_score": 30.0,
                    "auto_score": 30.0,
                    "sup_score": 10.0,
                    "col_score": 10.0,
                }.items()
                if k in df.columns
            },
        }
        df = self.classifier.categorize_scores_by_threshold(df, all_thresholds)
        Path(output_file).parent.mkdir(exist_ok=True, parents=True)
        df.to_csv(output_file, index=False, encoding="utf-8-sig")
        logger.info(f"CSV exporté → {output_file}")
        metrics = {
            "rows_initial": df_raw.shape[0],
            "rows_final": len(df),
            "columns_generated": df.shape[1],
            "thresholds": df.attrs.get("thresholds", {}),
        }
        return df, metrics

    def generate_visualizations(self, df):
        return self.plotter.generate_all_plots(df)

    def generate_crosstabs(self, df):
        return self.crosstab_gen.export_crosstabs(df, self.crosstab_list)


pipeline = KarasekPipeline()
print("Pipeline défini")


## 12. Exécution du pipeline

> **Modifiez** `Config.input_file` si votre fichier CSV est ailleurs.


In [ ]:
# ── Chargement + nettoyage + scores + classification ─────────────────────────
df_clean, metrics = pipeline.run(drop_missing=True)

print(f"Pipeline terminé")
print(f"   Lignes traitées : {metrics['rows_final']} / {metrics['rows_initial']}")
print(f"   Colonnes générées: {metrics['columns_generated']}")
print(f"   Seuils : {metrics['thresholds']}")


## 13. Aperçu des données nettoyées


In [ ]:
print(f"Shape : {df_clean.shape}")
df_clean.head()


In [ ]:
# Statistiques descriptives des scores continus
score_cols = [c for c in pipeline.score_cols if c in df_clean.columns]
df_clean[score_cols].describe().round(2)


## 14. Distributions des scores Karasek

Histogrammes + lignes de seuil théorique pour les 3 scores principaux.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
params = [
    ("Dem_score", config.DEM, "Demande Psychologique", "#d62728"),
    ("Lat_score", config.LAT, "Latitude Décisionnelle", "#1f77b4"),
    ("SS_score", config.SS, "Soutien Social", "#2ca02c"),
]
for ax, (col, threshold, label, color) in zip(axes, params):
    if col not in df_clean.columns:
        continue
    ax.hist(df_clean[col].dropna(), bins=20, color=color, alpha=0.7, edgecolor="white")
    ax.axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Seuil théorique = {threshold}",
    )
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xlabel("Score", fontsize=10)
    ax.set_ylabel("Effectif", fontsize=10)
    ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
plt.suptitle("Distribution des scores Karasek", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 15. Résultats de classification

### 15.1 Quadrants de Karasek (seuil théorique)


In [ ]:
if "Karasek_quadrant" in df_clean.columns:
    vc = df_clean["Karasek_quadrant"].value_counts()
    pct = (vc / len(df_clean) * 100).round(1)
    summary = pd.DataFrame({"Effectif": vc, "Pourcentage (%)": pct})
    display(summary)


In [ ]:
# Scatterplot Demande × Latitude
plotter.plot_karasek_scatter(df_clean)


### 15.2 Prévalence Job Strain et Iso Strain


In [ ]:
for col in ["Job_strain", "Iso_strain"]:
    if col not in df_clean.columns:
        continue
    n = (df_clean[col] == "Présent").sum()
    pct = n / len(df_clean) * 100
    print(f"  {col} — Présent : {pct:.1f}%  (n={n} / {len(df_clean)})")


## 16. Graphiques socio-démographiques

Barplots simples pour chaque variable catégorielle + pyramide des âges.


In [ ]:
# Pyramide des âges
plotter.plot_age_pyramid(df_clean)


In [ ]:
# Barplots simples — décommentez les colonnes qui existent dans votre dataset
vars_to_plot = [
    "Genre",
    "Situation matrimonial",
    "Tranche_age",
    "Tranche_anciennete",
    "Categorie_IMC",
    "Catégorie Socio",
]
for col in vars_to_plot:
    plotter.plot_categorical(df_clean, col)


## 17. Graphiques croisés (stacked bars)

Visualisation des tableaux croisés les plus importants.


In [ ]:
# Quadrants par genre et par tranche d'âge
for x_col, hue_col in [
    ("Genre", "Karasek_quadrant"),
    ("Tranche_age", "Karasek_quadrant"),
    ("Genre", "Job_strain"),
    ("Tranche_age", "Job_strain"),
]:
    plotter.plot_stacked_bar(df_clean, x_col, hue_col)


## 18. Heatmap Karasek par Direction


In [ ]:
plotter.plot_karasek_direction_heatmap(df_clean, group_col="Direction")


## 19. Scores RH

Répartition Faible / Élevé pour chaque dimension RH.


In [ ]:
rh_cat_cols = [
    f"{g}_score_cat"
    for g in config.RH_SCORE_GROUPS
    if f"{g}_score_cat" in df_clean.columns
]
for col in rh_cat_cols:
    plotter.plot_categorical(df_clean, col)


## 20. Exports

- CSV nettoyé enrichi
- Tableaux croisés Excel


In [ ]:
# Tableaux croisés Excel
ct_file = pipeline.generate_crosstabs(df_clean)
print(f"Tableaux croisés → {ct_file}")

# Rappel CSV
print(f"Données nettoyées → {config.output_file}")


## 21. Rapport texte de synthèse


In [ ]:
def generate_summary_report(df, metrics, output_file="../reports/karasek/resume.txt"):
    Path(output_file).parent.mkdir(exist_ok=True, parents=True)
    sep = "=" * 80
    lines = [
        f"{sep}\nRAPPORT D'ANALYSE KARASEK – COMPANY NAME\n{sep}",
        f"Date           : {datetime.now():%Y-%m-%d %H:%M:%S}",
        f"Lignes traitées: {metrics['rows_final']} / {metrics['rows_initial']}",
        f"Colonnes       : {metrics['columns_generated']}",
        "\n" + "-" * 80 + "\nSEUILS THÉORIQUES — POINT CENTRAL DE L'ÉCHELLE",
        "(Usage : dashboard — monitoring inter-vagues)",
        "  Libellé : 'niveau supérieur au point central' — ne pas écrire 'à risque'",
        "-" * 80,
    ]
    for k, v in metrics.get("thresholds", {}).items():
        if k not in ("method", "usage", "note"):
            lines.append(f"  {k}: {v}")
    lines.append(f"  Note : {metrics.get('thresholds', {}).get('note', '')}")

    lines += ["\n" + "-" * 80 + "\nSCORES KARASEK\n" + "-" * 80]
    for sc in ["Dem_score", "Lat_score", "SS_score"]:
        if sc in df.columns:
            s = df[sc]
            lbl = config.SCORE_LABELS.get(sc, sc)
            lines.append(
                f"  {lbl}: moy={s.mean():.2f} ± {s.std():.2f}  [{s.min():.1f}–{s.max():.1f}]"
            )

    lines += ["\n" + "-" * 80 + "\nQUADRANTS KARASEK (seuil théorique)\n" + "-" * 80]
    if "Karasek_quadrant" in df.columns:
        for q, pct in (
            df["Karasek_quadrant"].value_counts(normalize=True) * 100
        ).items():
            n = (df["Karasek_quadrant"] == q).sum()
            lines.append(f"  {q}: {pct:.1f}%  (n={n})")

    lines += ["\n" + "-" * 80 + "\nPRÉVALENCE RPS (seuil théorique)\n" + "-" * 80]
    for col in ["Job_strain", "Iso_strain"]:
        if col in df.columns:
            n = (df[col] == "Présent").sum()
            lines.append(f"  {col}: {n/len(df)*100:.1f}%  (n={n})")

    lines.append(f"\n{sep}\nFIN DU RAPPORT\n{sep}")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Rapport → {output_file}")
    return output_file


generate_summary_report(df_clean, metrics)


## Rappel méthodologique final

| Symbole | Méthode       | Seuil                   | Usage                               |
| ------- | ------------- | ----------------------- | ----------------------------------- |
| `*`     | **Théorique** | Point central (N × 2.5) | Dashboard — monitoring inter-vagues |

---

<br>

> - **Toujours conserver les scores continus en base** — ne jamais sauvegarder uniquement les catégories
> - **Ne jamais écrire « à risque »** — utiliser « niveau supérieur au point central de l'échelle »
> - Les pourcentages de Job Strain sont basés sur le **seuil théorique**
